# Deno project configuration lab

Inspect the Deno runtime used by Spur Notebook and determine which project configuration notebook cells actually see. Starting project: `/Users/kevintruong/project/spur-notebook`.

This notebook performs diagnostics only. It does not change project configuration, install project dependencies, run package scripts or expose credential values.

Questions:

1. Which Deno version, executable, working directory and permissions does the notebook use?
2. Which `deno.json`, `deno.jsonc`, `package.json`, import-map and lock files exist at the project root or ancestor locations?
3. Does the notebook honor a project import alias, compiler/JSX settings and configuration discovery, or does the kernel use separate startup settings?
4. What differs between a standalone Deno command and a notebook cell?

The cells retain their observed results. Project changes should be tested in an isolated fixture rather than editing the existing app.

In [2]:
// Runtime facts only. No environment or process argument dump.
const denoProjectProbe = {projectRoot: '/Users/kevintruong/project/spur-notebook', notebookPath: '/Users/kevintruong/project/spur-notebook/2026-11-18-deno-project-configuration.ipynb'};
const permissionRows = [];
for (const name of ['read','write','net','run','env','sys','ffi']) {
  try { permissionRows.push({name,state:(await Deno.permissions.query({name})).state}); }
  catch(e) { permissionRows.push({name,state:'unsupported',error:e.name}); }
}
denoProjectProbe.runtime = {versions:Deno.version,executable:Deno.execPath(),cwd:Deno.cwd(),build:Deno.build,permissions:permissionRows};
console.log(JSON.stringify(denoProjectProbe.runtime,null,2));

{
  "versions": {
    "deno": "2.8.1",
    "v8": "14.9.207.2-rusty",
    "typescript": "6.0.3"
  },
  "executable": "/Users/kevintruong/Applications/SpurLab.app/Contents/MacOS/deno",
  "cwd": "/Users/kevintruong/project/spur-notebook",
  "build": {
    "target": "aarch64-apple-darwin",
    "arch": "aarch64",
    "os": "darwin",
    "vendor": "apple",
    "standalone": false
  },
  "permissions": [
    {
      "name": "read",
      "state": "granted"
    },
    {
      "name": "write",
      "state": "granted"
    },
    {
      "name": "net",
      "state": "granted"
    },
    {
      "name": "run",
      "state": "granted"
    },
    {
      "name": "env",
      "state": "granted"
    },
    {
      "name": "sys",
      "state": "granted"
    },
    {
      "name": "ffi",
      "state": "granted"
    }
  ]
}


## Project files and discovery candidates

The next cell checks named configuration files at the project root, the kernel working directory and their ancestors. It reports only safe summaries: config keys, dependency/import names, task names and compiler settings. It does not print environment values, npm configuration, task commands or authenticated import URLs.

File presence is **not** proof that the running notebook kernel loaded that file. A later isolated fixture tests configuration behavior.

In [3]:
const configNames = ['deno.json','deno.jsonc','package.json','deno.lock','import_map.json'];
const candidateDirs = new Set();
for (const root of [denoProjectProbe.projectRoot,Deno.cwd()]) {
  let current=root;
  for(let depth=0;depth<8;depth++) {candidateDirs.add(current);const next=current.slice(0,current.lastIndexOf('/'))||'/';if(next===current)break;current=next;}
}
const configRows=[];
for(const dir of candidateDirs) for(const name of configNames){
  const path=dir.replace(/\/$/,'')+'/'+name;
  try {
    const stat=await Deno.stat(path);if(!stat.isFile)continue;
    const row={path,bytes:stat.size};
    if(stat.size>512000){row.summary='File present; contents not read (diagnostic size bound)';configRows.push(row);continue;}
    const text=await Deno.readTextFile(path);
    try {
      const cfg=JSON.parse(text);row.json='valid';row.keys=Object.keys(cfg);
      if(name==='package.json'){
        row.package={name:cfg.name,version:cfg.version,type:cfg.type,packageManager:cfg.packageManager,dependencyNames:Object.keys(cfg.dependencies||{}),devDependencyCount:Object.keys(cfg.devDependencies||{}).length,scriptNames:Object.keys(cfg.scripts||{})};
      } else if(name==='deno.lock'){
        row.lock={version:cfg.version,specifierCount:Object.keys(cfg.specifiers||{}).length,npmCount:Object.keys(cfg.npm||{}).length,jsrCount:Object.keys(cfg.jsr||{}).length,remoteCount:Object.keys(cfg.remote||{}).length};
      } else {
        row.config={importNames:Object.keys(cfg.imports||{}),scopeNamesCount:Object.keys(cfg.scopes||{}).length,taskNames:Object.keys(cfg.tasks||{}),nodeModulesDir:cfg.nodeModulesDir,lock:typeof cfg.lock==='boolean'?cfg.lock:typeof cfg.lock,compilerOptions:cfg.compilerOptions?{jsx:cfg.compilerOptions.jsx,jsxImportSource:cfg.compilerOptions.jsxImportSource,strict:cfg.compilerOptions.strict,lib:cfg.compilerOptions.lib}:undefined,workspaceMembers:cfg.workspace?.length,externalImportMap:typeof cfg.importMap==='string'};
      }
    }catch(e){row.json='Not strict JSON; may be valid JSONC. Needs Deno CLI validation.';}
    configRows.push(row);
  }catch(e){if(e.name!=='NotFound')configRows.push({path,error:e.name});}
}
denoProjectProbe.inventory=configRows;
console.log(JSON.stringify({projectRoot:denoProjectProbe.projectRoot,kernelCwd:Deno.cwd(),candidateDirectoryCount:candidateDirs.size,found:configRows},null,2));

{
  "projectRoot": "/Users/kevintruong/project/spur-notebook",
  "kernelCwd": "/Users/kevintruong/project/spur-notebook",
  "candidateDirectoryCount": 5,
  "found": [
    {
      "path": "/Users/kevintruong/package.json",
      "bytes": 62,
      "json": "valid",
      "keys": [
        "dependencies"
      ],
      "package": {
        "dependencyNames": [
          "@getspur/spur-cli"
        ],
        "devDependencyCount": 0,
        "scriptNames": []
      }
    }
  ]
}


## Isolated project fixture

Create a temporary local-only Deno project with an import alias, a TypeScript module, a JSON import, a custom JSX runtime and a small TSX entrypoint. Compare explicit/automatic CLI configuration with loading the same entrypoint in this already-running notebook kernel.

The fixture uses no third-party packages and makes no network requests. CLI runs use cached-only resolution and disable lockfile writes. No project scripts are executed. Changing the current directory inside the notebook, when probed, is always restored.

In [4]:
denoProjectProbe.fixtureDir = await Deno.makeTempDir({dir:'/private/tmp',prefix:'spur-deno-config-'});
const fixtureFiles = {
  'deno.json': JSON.stringify({imports:{'@config-probe':'./module.ts','@probe-jsx/jsx-runtime':'./jsx-runtime.ts'},compilerOptions:{strict:true,jsx:'react-jsx',jsxImportSource:'@probe-jsx'},nodeModulesDir:'none',lock:false},null,2),
  'module.ts': 'export const configuredValue: string = "local-alias-ok";\n',
  'data.json': JSON.stringify({message:'json-import-ok'}),
  'entry.ts': 'import { configuredValue } from "@config-probe";\nimport data from "./data.json" with { type: "json" };\nexport const result = {alias:configuredValue,json:data.message};\nconsole.log(JSON.stringify(result));\n',
  'jsx-runtime.ts': 'export namespace JSX { export interface IntrinsicElements { article: {children?: unknown}; } }\nexport const Fragment = "fragment";\nexport function jsx(type: string, props: unknown) { return {type,props}; }\nexport const jsxs = jsx;\n',
  'entry.tsx': 'import { configuredValue } from "@config-probe";\nconst result = <article>{configuredValue}</article>;\nconsole.log(JSON.stringify(result));\nexport { result };\n'
};
for (const [name,source] of Object.entries(fixtureFiles)) await Deno.writeTextFile(denoProjectProbe.fixtureDir+'/'+name,source);
console.log(JSON.stringify({fixtureDir:denoProjectProbe.fixtureDir,files:Object.keys(fixtureFiles),projectFilesChanged:false,externalDependencies:0},null,2));

{
  "fixtureDir": "/private/tmp/spur-deno-config-f532f4ef4eed273",
  "files": [
    "deno.json",
    "module.ts",
    "data.json",
    "entry.ts",
    "jsx-runtime.ts",
    "entry.tsx"
  ],
  "projectFilesChanged": false,
  "externalDependencies": 0
}


In [5]:
async function runDenoConfigProbe(label,args,cwd){
  const command=new Deno.Command(Deno.execPath(),{args,cwd,stdout:'piped',stderr:'piped'});
  const child=command.spawn();
  const timer=setTimeout(()=>{try{child.kill('SIGTERM')}catch{}},15000);
  try{const out=await child.output();return {label,args,exitCode:out.code,stdout:new TextDecoder().decode(out.stdout).trim(),stderr:new TextDecoder().decode(out.stderr).replace(/\x1b\[[0-9;]*m/g,'').slice(0,1800).trim()};}
  finally{clearTimeout(timer);}
}
const fixture=denoProjectProbe.fixtureDir;
denoProjectProbe.cli=[];
for(const [label,args,cwd] of [
  ['Explicit config: alias + JSON',['run','--cached-only','--no-lock','--no-prompt','--config',fixture+'/deno.json',fixture+'/entry.ts'],fixture],
  ['Automatic config: alias + JSON',['run','--cached-only','--no-lock','--no-prompt','entry.ts'],fixture],
  ['Explicit config: JSX runtime',['run','--cached-only','--no-lock','--no-prompt','--config',fixture+'/deno.json',fixture+'/entry.tsx'],fixture],
  ['Explicit config: type check',['check','--cached-only','--no-lock','--config',fixture+'/deno.json',fixture+'/entry.tsx'],fixture],
  ['Control: configuration disabled',['run','--cached-only','--no-lock','--no-prompt','--no-config',fixture+'/entry.ts'],fixture]
]) denoProjectProbe.cli.push(await runDenoConfigProbe(label,args,cwd));
console.log(JSON.stringify(denoProjectProbe.cli,null,2));

[
  {
    "label": "Explicit config: alias + JSON",
    "args": [
      "run",
      "--cached-only",
      "--no-lock",
      "--no-prompt",
      "--config",
      "/private/tmp/spur-deno-config-f532f4ef4eed273/deno.json",
      "/private/tmp/spur-deno-config-f532f4ef4eed273/entry.ts"
    ],
    "exitCode": 0,
    "stdout": "{\"alias\":\"local-alias-ok\",\"json\":\"json-import-ok\"}",
    "stderr": ""
  },
  {
    "label": "Automatic config: alias + JSON",
    "args": [
      "run",
      "--cached-only",
      "--no-lock",
      "--no-prompt",
      "entry.ts"
    ],
    "exitCode": 0,
    "stdout": "{\"alias\":\"local-alias-ok\",\"json\":\"json-import-ok\"}",
    "stderr": ""
  },
  {
    "label": "Explicit config: JSX runtime",
    "args": [
      "run",
      "--cached-only",
      "--no-lock",
      "--no-prompt",
      "--config",
      "/private/tmp/spur-deno-config-f532f4ef4eed273/deno.json",
      "/private/tmp/spur-deno-config-f532f4ef4eed273/entry.tsx"
    ],
    "exitCode

In [6]:
// Compare the running notebook kernel with the CLI fixture, without restarting or editing project config.
const notebookImportRows=[];
async function captureNotebookImport(label,load){
 try{const mod=await load();return {label,status:'loaded',result:mod.result??mod.configuredValue??mod.default};}
 catch(e){return {label,status:'not loaded',error:e.name,message:String(e.message).slice(0,900)};}
}
notebookImportRows.push(await captureNotebookImport('Direct file URL, no alias',()=>import('file://'+denoProjectProbe.fixtureDir+'/module.ts')));
notebookImportRows.push(await captureNotebookImport('Direct JSON file URL',()=>import('file://'+denoProjectProbe.fixtureDir+'/data.json',{with:{type:'json'}})));
notebookImportRows.push(await captureNotebookImport('Configured fixture entrypoint from notebook',()=>import('file://'+denoProjectProbe.fixtureDir+'/entry.ts')));
const originalCwd=Deno.cwd();
try{
 Deno.chdir(denoProjectProbe.fixtureDir);
 notebookImportRows.push(await captureNotebookImport('Import alias after chdir to configured fixture',()=>import('@config-probe')));
}finally{Deno.chdir(originalCwd);}
denoProjectProbe.notebookImports=notebookImportRows;
console.log(JSON.stringify({imports:notebookImportRows,cwdRestored:Deno.cwd()===originalCwd,currentCwd:Deno.cwd()},null,2));
const checkHelp=await runDenoConfigProbe('Installed deno check flag discovery',['check','--help'],denoProjectProbe.fixtureDir);
denoProjectProbe.checkHelp={exitCode:checkHelp.exitCode,relevantLines:checkHelp.stdout.split('\n').filter(line=>/cached|remote|npm|lock|config|Type-check|Usage:/.test(line))};
console.log(JSON.stringify(denoProjectProbe.checkHelp,null,2));

{
  "imports": [
    {
      "label": "Direct file URL, no alias",
      "status": "loaded",
      "result": "local-alias-ok"
    },
    {
      "label": "Direct JSON file URL",
      "status": "loaded",
      "result": {
        "message": "json-import-ok"
      }
    },
    {
      "label": "Configured fixture entrypoint from notebook",
      "status": "not loaded",
      "error": "TypeError",
      "message": "Import \"@config-probe\" not a dependency\n    at \u001b\u001bfile:///private/tmp/spur-deno-config-f532f4ef4eed273/entry.ts\u001b:\u001b\u001b1\u001b:\u001b\u001b33\u001b"
    },
    {
      "label": "Import alias after chdir to configured fixture",
      "status": "not loaded",
      "error": "TypeError",
      "message": "Import \"@config-probe\" not a dependency"
    }
  ],
  "cwdRestored": true,
  "currentCwd": "/Users/kevintruong/project/spur-notebook"
}


{
  "exitCode": 0,
  "relevantLines": [
    "Unless --reload is specified, this command will not re-download already cached dependencies",
    "Usage: deno check [OPTIONS] [file]...",
    "      --all                                              Type-check all code, including remote modules and npm packages",
    "  -I, --allow-import[=<IP_OR_HOSTNAME>...]               Allow importing from remote hosts. Optionally specify allowed IP addresses and host names, with ports as necessary. Default value: deno.land:443,jsr.io:443,esm.sh:443,raw.esm.sh:443,cdn.jsdelivr.net:443,raw.githubusercontent.com:443,gist.githubusercontent.com:443",
    "      --conditions <conditions>                          Use this argument to specify custom conditions for npm package exports. You can also use DENO_CONDITIONS env var.",
    "  -c, --config <FILE>                                    Configure different aspects of deno including TypeScript, linting, and code formatting.",
    "                          

## Validate the observed boundary

The CLI fixture loaded its local alias and custom JSX runtime. Loading the configured entrypoint in the already-running notebook did not resolve that alias, including after `Deno.chdir()`; direct file/JSON imports did work.

The first `deno check` trial used an unsupported `--cached-only` flag. This is a probe-command error, not a failed project type check. The installed `deno check --help` advertises `--no-remote` and `--no-npm`; the next trial uses those local-only flags instead. The original trial is retained above.

In [7]:
denoProjectProbe.typecheck = await runDenoConfigProbe('Local-only configured type check',['check','--no-remote','--no-npm','--no-lock','--config',denoProjectProbe.fixtureDir+'/deno.json',denoProjectProbe.fixtureDir+'/entry.tsx'],denoProjectProbe.fixtureDir);
const jupyterHelp = await runDenoConfigProbe('Installed Deno Jupyter startup flags',['jupyter','--help'],denoProjectProbe.projectRoot);
denoProjectProbe.jupyterFlags = {exitCode:jupyterHelp.exitCode,hasConfigFlag:/--config\b/.test(jupyterHelp.stdout),hasImportMapFlag:/--import-map\b/.test(jupyterHelp.stdout),relevantLines:jupyterHelp.stdout.split('\n').filter(line=>/Usage:|--config|--import-map|--kernel|--conn|--install|--cwd/.test(line))};
const conditions = [
  {check:'CLI explicit alias + JSON',passed:denoProjectProbe.cli[0].exitCode===0},
  {check:'CLI automatic config discovery',passed:denoProjectProbe.cli[1].exitCode===0},
  {check:'CLI configured JSX runtime',passed:denoProjectProbe.cli[2].exitCode===0},
  {check:'Corrected local-only type check',passed:denoProjectProbe.typecheck.exitCode===0},
  {check:'No-config control rejects alias',passed:denoProjectProbe.cli[4].exitCode!==0&&denoProjectProbe.cli[4].stderr.includes('@config-probe')},
  {check:'Notebook direct file module import works',passed:denoProjectProbe.notebookImports[0].status==='loaded'},
  {check:'Notebook direct JSON import works',passed:denoProjectProbe.notebookImports[1].status==='loaded'},
  {check:'Existing notebook does not discover fixture alias',passed:denoProjectProbe.notebookImports[2].status==='not loaded'},
  {check:'Changing cwd does not activate fixture alias',passed:denoProjectProbe.notebookImports[3].status==='not loaded'},
  {check:'Notebook working directory restored',passed:Deno.cwd()===denoProjectProbe.projectRoot},
];
denoProjectProbe.verification={status:conditions.every(x=>x.passed)?'observations confirmed':'review needed',checks:conditions};
console.log(JSON.stringify({typecheck:denoProjectProbe.typecheck,jupyterStartup:denoProjectProbe.jupyterFlags,verification:denoProjectProbe.verification},null,2));

{
  "typecheck": {
    "label": "Local-only configured type check",
    "args": [
      "check",
      "--no-remote",
      "--no-npm",
      "--no-lock",
      "--config",
      "/private/tmp/spur-deno-config-f532f4ef4eed273/deno.json",
      "/private/tmp/spur-deno-config-f532f4ef4eed273/entry.tsx"
    ],
    "exitCode": 0,
    "stdout": "",
    "stderr": "Check entry.tsx"
  },
  "jupyterStartup": {
    "exitCode": 0,
    "hasConfigFlag": false,
    "hasImportMapFlag": false,
    "relevantLines": [
      "Usage: deno jupyter [OPTIONS]",
      "      --conn <conn>        Path to JSON file describing connection parameters, provided by Jupyter",
      "      --install            Install a kernelspec",
      "      --kernel             Start the kernel"
    ]
  },
  "verification": {
    "status": "observations confirmed",
    "checks": [
      {
        "check": "CLI explicit alias + JSON",
        "passed": true
      },
      {
        "check": "CLI automatic config discovery",
      

## Findings

| Area | Observed result |
|---|---|
| Notebook runtime | Deno 2.8.1, TypeScript 6.0.3, bundled with SpurLab, Apple arm64 |
| Notebook working directory | `/Users/kevintruong/project/spur-notebook` |
| Existing Deno project config | No `deno.json` / `deno.jsonc` found at the working directory or its ancestors |
| Existing package metadata | Only `/Users/kevintruong/package.json` found in the inspected ancestor chain; this does not prove the kernel loaded it |
| Direct local module and JSON imports | Both loaded in notebook cells |
| Isolated CLI project | Explicit config and automatic config discovery both resolved the local alias |
| Compiler settings | Custom JSX runtime worked; corrected local-only `deno check` passed |
| Existing notebook kernel | Importing the fixture entrypoint did not discover its alias; `Deno.chdir()` did not activate it |
| Installed Jupyter command | Help advertises neither `--config` nor `--import-map`; this is a help-surface observation, not a complete startup capability audit |
| Verification | 10 observations confirmed; expected negative import cases are recorded, not hidden |

### What this means for a notebook app

Treat the project root, Deno configuration and running kernel state as separate settings. Moving a notebook's working directory is not evidence that import mappings or compiler settings changed. Direct file imports are a verified option in this session. A configured CLI project is also verified, but is a separate execution context.

Before proposing an app change, test a **fresh kernel launched with a pre-existing configuration in its startup directory**, then inspect the notebook host's project/config controls and startup implementation. This notebook has not tested fresh configured-kernel startup, package installation, dependency reproducibility, a JSX expression directly in a cell, or live config reload. It does not establish that all Deno project configuration is unsupported.

The temporary fixture remains available for inspection at the path printed above. No existing project config was edited, and the working directory was restored.

References: [Deno configuration discovery and configuration fields](https://docs.deno.com/runtime/fundamentals/configuration/), [Deno Jupyter kernel](https://docs.deno.com/runtime/reference/cli/jupyter/). Installed-binary behavior and saved cell outputs take precedence over general documentation for this environment.

In [8]:
// Save a safe, reproducible diagnostics receipt. Existing project config is not modified.
const receiptDirectory='/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-18-deno-project-configuration';
await Deno.mkdir(receiptDirectory,{recursive:true});
const fixtureHashes=[];
for(const name of Object.keys(fixtureFiles)){
 const bytes=await Deno.readFile(denoProjectProbe.fixtureDir+'/'+name);
 const hash=Array.from(new Uint8Array(await crypto.subtle.digest('SHA-256',bytes))).map(v=>v.toString(16).padStart(2,'0')).join('');
 fixtureHashes.push({name,sha256:hash});
}
const receipt={schemaVersion:1,notebookPath:denoProjectProbe.notebookPath,projectRoot:denoProjectProbe.projectRoot,runtime:denoProjectProbe.runtime,inventory:denoProjectProbe.inventory,fixtureDirectory:denoProjectProbe.fixtureDir,fixtureHashes,cli:denoProjectProbe.cli,correctedTypecheck:denoProjectProbe.typecheck,notebookImports:denoProjectProbe.notebookImports,jupyterStartup:denoProjectProbe.jupyterFlags,verification:denoProjectProbe.verification,limits:['Fresh configured-kernel startup not tested','Existing project config unchanged','No dependency installation or network import probe','No host implementation inspected']};
await Deno.writeTextFile(receiptDirectory+'/results.json',JSON.stringify(receipt,null,2)+'\n');
await Deno.mkdir(receiptDirectory+'/fixture',{recursive:true});
for(const name of Object.keys(fixtureFiles)) await Deno.copyFile(denoProjectProbe.fixtureDir+'/'+name,receiptDirectory+'/fixture/'+name);
console.log(JSON.stringify({saved:receiptDirectory,status:receipt.verification.status,checks:receipt.verification.checks.length,cwd:Deno.cwd()},null,2));

{
  "saved": "/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-18-deno-project-configuration",
  "status": "observations confirmed",
  "checks": 10,
  "cwd": "/Users/kevintruong/project/spur-notebook"
}
